In [68]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout,LSTM,GRU,Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error,r2_score

In [29]:
df_train = pd.read_csv(r'../data/cleaned/train.csv')
df_test = pd.read_csv(r'../data/cleaned/test.csv')
df_val = pd.read_csv(r'../data/cleaned/val.csv')

In [30]:
with open('target_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

In [31]:
df_train

,Datetime,PJME_MW,year,month,hour,day_num,day_of_month,week_of_year,is_weekend,lag_1,lag_24,lag_168,rolling_mean_24,rolling_std_24
0,2002-01-09 01:00:00,0.306289,2002,1,1,2,9,2,0,0.345497,0.313937,0.286042,0.456556,0.234076
1,2002-01-09 02:00:00,0.286738,2002,1,2,2,9,2,0,0.306289,0.297609,0.271632,0.456097,0.236377
2,2002-01-09 03:00:00,0.279890,2002,1,3,2,9,2,0,0.286738,0.291394,0.268766,0.455444,0.240153
3,2002-01-09 04:00:00,0.278416,2002,1,4,2,9,2,0,0.279890,0.294912,0.273654,0.454754,0.244299
4,2002-01-09 05:00:00,0.289982,2002,1,5,2,9,2,0,0.278416,0.310060,0.292026,0.453764,0.250095
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101635,2013-08-13 20:00:00,0.560813,2013,8,20,1,13,33,0,0.592205,0.599199,0.419973,0.532663,0.430098
101636,2013-08-13 21:00:00,0.558074,2013,8,21,1,13,33,0,0.560813,0.590393,0.431118,0.530359,0.422228
101637,2013-08-13 22:00:00,0.517455,2013,8,22,1,13,33,0,0.558074,0.551796,0.412262,0.528420,0.415776
101638,2013-08-13 23:00:00,0.449342,2013,8,23,1,13,33,0,0.517455,0.477594,0.363194,0.526359,0.411341


In [32]:
df_test

,Datetime,PJME_MW,year,month,hour,day_num,day_of_month,week_of_year,is_weekend,lag_1,lag_24,lag_168,rolling_mean_24,rolling_std_24
0,2016-02-07 13:00:00,0.327568,2016,2,13,6,7,5,1,0.336248,0.337112,0.278205,0.335566,0.048513
1,2016-02-07 14:00:00,0.320615,2016,2,14,6,7,5,1,0.327568,0.320510,0.265606,0.334993,0.048556
2,2016-02-07 15:00:00,0.317244,2016,2,15,6,7,5,1,0.320615,0.309913,0.257095,0.335000,0.048548
3,2016-02-07 16:00:00,0.323480,2016,2,16,6,7,5,1,0.317244,0.306900,0.262256,0.335440,0.047735
4,2016-02-07 17:00:00,0.343811,2016,2,17,6,7,5,1,0.323480,0.322638,0.283451,0.336435,0.045987
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21775,2018-08-02 20:00:00,0.621784,2018,8,20,3,2,31,0,0.655156,0.681934,0.669820,0.628587,0.532395
21776,2018-08-02 21:00:00,0.604909,2018,8,21,3,2,31,0,0.621784,0.662404,0.632003,0.624977,0.521400
21777,2018-08-02 22:00:00,0.569009,2018,8,22,3,2,31,0,0.604909,0.622564,0.591889,0.621527,0.512173
21778,2018-08-02 23:00:00,0.504709,2018,8,23,3,2,31,0,0.569009,0.550342,0.521058,0.618313,0.506551


In [33]:
df_val

,Datetime,PJME_MW,year,month,hour,day_num,day_of_month,week_of_year,is_weekend,lag_1,lag_24,lag_168,rolling_mean_24,rolling_std_24
0,2013-08-14 01:00:00,0.319751,2013,8,1,2,14,33,0,0.379374,0.349141,0.263647,0.522917,0.415302
1,2013-08-14 02:00:00,0.274055,2013,8,2,2,14,33,0,0.319751,0.311767,0.234952,0.521153,0.422386
2,2013-08-14 03:00:00,0.243148,2013,8,3,2,14,33,0,0.274055,0.290909,0.217950,0.518890,0.434119
3,2013-08-14 04:00:00,0.223891,2013,8,4,2,14,33,0,0.243148,0.280733,0.211904,0.516024,0.450687
4,2013-08-14 05:00:00,0.218645,2013,8,5,2,14,33,0,0.223891,0.289729,0.220752,0.512612,0.471038
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21775,2016-02-07 08:00:00,0.347898,2016,2,8,6,7,5,1,0.329864,0.381839,0.309891,0.344615,0.072727
21776,2016-02-07 09:00:00,0.359654,2016,2,9,6,7,5,1,0.347898,0.398820,0.325103,0.342579,0.068262
21777,2016-02-07 10:00:00,0.355019,2016,2,10,6,7,5,1,0.359654,0.388560,0.322996,0.340228,0.059683
21778,2016-02-07 11:00:00,0.344823,2016,2,11,6,7,5,1,0.355019,0.371411,0.306268,0.338215,0.053128


In [34]:
def create_sequences(df, target_col='PJME_MW', seq_length=24, horizon=1):
    """
    Converts a scaled DataFrame into 3D sequence arrays for Keras models.
    """
    # Separate feature columns from Datetime
    feature_cols = [col for col in df.columns if col != 'Datetime']
    target_idx = feature_cols.index(target_col)
    
    data = df[feature_cols].values
    X, y = [], []
    
    for i in range(len(data) - seq_length - horizon + 1):
        X.append(data[i : i + seq_length, :])
        if horizon == 1:
            y.append(data[i + seq_length, target_idx])
        else:
            y.append(data[i + seq_length : i + seq_length + horizon, target_idx])
            
    return np.array(X), np.array(y)

In [54]:
SEQ_LEN = 168
HORIZON = 1   

X_train, y_train = create_sequences(df_train, seq_length=SEQ_LEN, horizon=HORIZON)
X_val, y_val     = create_sequences(df_val, seq_length=SEQ_LEN, horizon=HORIZON)
X_test, y_test   = create_sequences(df_test, seq_length=SEQ_LEN, horizon=HORIZON)

print(f"X_train Shape: {X_train.shape}") # (samples, 24, num_features)
print(f"X_test Shape:  {X_test.shape}")

X_train Shape: (101472, 168, 13)
X_test Shape:  (21612, 168, 13)


In [56]:
input_shape = (X_train.shape[1], X_train.shape[2])

# Build Basic RNN
model_rnn = Sequential([
    SimpleRNN(64, return_sequences=False, input_shape=input_shape),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_rnn.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_rnn = model_rnn.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=50,batch_size=64,verbose=1)

Epoch 1/50
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 25s 14ms/step - loss: 0.0423 - mae: 0.1436 - val_loss: 0.0185 - val_mae: 0.1118
Epoch 2/50
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.0197 - mae: 0.1084 - val_loss: 0.0184 - val_mae: 0.1115
Epoch 3/50
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0196 - mae: 0.1082 - val_loss: 0.0170 - val_mae: 0.1040
Epoch 4/50
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0195 - mae: 0.1077 - val_loss: 0.0170 - val_mae: 0.1031
Epoch 5/50
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.0194 - mae: 0.1076 - val_loss: 0.0175 - val_mae: 0.1075
Epoch 6/50
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - loss: 0.0193 - mae: 0.1072 - val_loss: 0.0177 - val_mae: 0.1082
Epoch 7/50
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.0193 - mae: 0.1071 - val_loss: 0.0196 - val_mae: 0.1163
Epoch 8/50
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - loss: 0.0192 - mae: 0.1067 - val_loss: 0.0174 - val_mae: 0.1069
Epoch 9/50
1586/1586 ━━━

In [57]:
y_pred_rnn_scaled = model_rnn.predict(X_test)

# 2. Inverse-transform predictions and targets back to MW
y_pred_rnn_mw = scaler.inverse_transform(y_pred_rnn_scaled)
y_test_mw = scaler.inverse_transform(y_test.reshape(-1, 1))

# 3. Compute Metrics
mae_rnn = mean_absolute_error(y_test_mw, y_pred_rnn_mw)
rmse_rnn = np.sqrt(mean_squared_error(y_test_mw, y_pred_rnn_mw))
mape_rnn = np.mean(np.abs((y_test_mw - y_pred_rnn_mw) / y_test_mw)) * 100
r2_rnn = r2_score(y_test_mw, y_pred_rnn_mw)
print("--- Basic RNN Test Results ---")
print(f"MAE:  {mae_rnn:.2f} MW")
print(f"RMSE: {rmse_rnn:.2f} MW")
print(f"MAPE: {mape_rnn:.2f}%")
print(f"r2: {r2_rnn:.2f}%")

676/676 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step
--- Basic RNN Test Results ---
MAE:  5225.23 MW
RMSE: 6535.00 MW
MAPE: 17.63%
r2: -0.03%


In [ ]:
#lstm

In [59]:
model_lstm = Sequential([
    LSTM(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_lstm.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_lstm = model_lstm.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=20,batch_size=64,verbose=1)

Epoch 1/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 69s 42ms/step - loss: 0.0226 - mae: 0.1152 - val_loss: 0.0175 - val_mae: 0.1026
Epoch 2/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 66s 41ms/step - loss: 0.0193 - mae: 0.1072 - val_loss: 0.0170 - val_mae: 0.1034
Epoch 3/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 65s 41ms/step - loss: 0.0192 - mae: 0.1070 - val_loss: 0.0172 - val_mae: 0.1026
Epoch 4/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 66s 41ms/step - loss: 0.0191 - mae: 0.1065 - val_loss: 0.0201 - val_mae: 0.1177
Epoch 5/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 66s 41ms/step - loss: 0.0191 - mae: 0.1067 - val_loss: 0.0170 - val_mae: 0.1029
Epoch 6/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 66s 41ms/step - loss: 0.0191 - mae: 0.1066 - val_loss: 0.0171 - val_mae: 0.1028
Epoch 7/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 65s 41ms/step - loss: 0.0191 - mae: 0.1064 - val_loss: 0.0179 - val_mae: 0.1092
Epoch 8/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 66s 41ms/step - loss: 0.0190 - mae: 0.1063 - val_loss: 0.0178 - val_mae: 0.1088
Epoch 9/20
1586/1586 ━━━

In [60]:
y_pred_lstm_scaled = model_lstm.predict(X_test)

676/676 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step


In [61]:
y_pred_lstm_mw = scaler.inverse_transform(y_pred_lstm_scaled.reshape(-1, 1))

In [64]:
mae_lstm = mean_absolute_error(y_test_mw, y_pred_lstm_mw)
rmse_lstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_lstm_mw))
mape_lstm = np.mean(np.abs((y_test_mw - y_pred_lstm_mw) / y_test_mw)) * 100
r2_lstm = r2_score(y_test_mw, y_pred_lstm_mw)

print(f"MAE:  {mae_lstm:.2f} MW")
print(f"RMSE: {rmse_lstm:.2f} MW")
print(f"MAPE: {mape_lstm:.2f}%")
print(f"R2: {r2_lstm:.2f}%")

MAE:  5117.05 MW
RMSE: 6480.64 MW
MAPE: 17.07%
R2: -0.01%


In [ ]:
#GRU

In [66]:
model_gru = Sequential([
    GRU(64, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_gru.compile(optimizer='adam', loss='mse', metrics=['mae'])


history_gru = model_gru.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=64,
    verbose=1
)

Epoch 1/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 101s 61ms/step - loss: 0.0214 - mae: 0.1126 - val_loss: 0.0170 - val_mae: 0.1029
Epoch 2/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 90s 57ms/step - loss: 0.0194 - mae: 0.1076 - val_loss: 0.0170 - val_mae: 0.1036
Epoch 3/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 77s 49ms/step - loss: 0.0193 - mae: 0.1073 - val_loss: 0.0201 - val_mae: 0.1177
Epoch 4/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 77s 49ms/step - loss: 0.0193 - mae: 0.1072 - val_loss: 0.0171 - val_mae: 0.1046
Epoch 5/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 77s 49ms/step - loss: 0.0192 - mae: 0.1070 - val_loss: 0.0180 - val_mae: 0.1097
Epoch 6/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 76s 48ms/step - loss: 0.0192 - mae: 0.1069 - val_loss: 0.0172 - val_mae: 0.1056
Epoch 7/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 76s 48ms/step - loss: 0.0191 - mae: 0.1066 - val_loss: 0.0179 - val_mae: 0.1093
Epoch 8/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 76s 48ms/step - loss: 0.0191 - mae: 0.1063 - val_loss: 0.0170 - val_mae: 0.1033
Epoch 9/20
1586/1586 ━━

In [67]:
# 1. Predict on scaled test set
y_pred_gru_scaled = model_gru.predict(X_test)

# 2. Inverse-transform back to real Megawatts
y_pred_gru_mw = scaler.inverse_transform(y_pred_gru_scaled.reshape(-1, 1))


# 3. Compute Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

mae_gru = mean_absolute_error(y_test_mw, y_pred_gru_mw)
rmse_gru = np.sqrt(mean_squared_error(y_test_mw, y_pred_gru_mw))
mape_gru = np.mean(np.abs((y_test_mw - y_pred_gru_mw) / y_test_mw)) * 100
r2_gru = r2_score(y_test_mw, y_pred_gru_mw)

print("--- GRU Test Results ---")
print(f"MAE:  {mae_gru:.2f} MW")
print(f"RMSE: {rmse_gru:.2f} MW")
print(f"MAPE: {mape_gru:.2f}%")
print(f"R2:   {r2_gru:.4f}")

676/676 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step
--- GRU Test Results ---
MAE:  5330.05 MW
RMSE: 6599.56 MW
MAPE: 18.14%
R2:   -0.0485


In [ ]:
#bidirectional lstm

In [70]:
model_bilstm = Sequential([
    Bidirectional(LSTM(64, return_sequences=False), input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_bilstm.compile(optimizer='adam', loss='mse', metrics=['mae'])


# 3. Train BiLSTM
history_bilstm = model_bilstm.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=64,
    verbose=1
)

Epoch 1/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 113s 69ms/step - loss: 0.0234 - mae: 0.1169 - val_loss: 0.0176 - val_mae: 0.1028
Epoch 2/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 106s 67ms/step - loss: 0.0197 - mae: 0.1085 - val_loss: 0.0172 - val_mae: 0.1026
Epoch 3/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 107s 67ms/step - loss: 0.0196 - mae: 0.1083 - val_loss: 0.0170 - val_mae: 0.1043
Epoch 4/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 106s 67ms/step - loss: 0.0195 - mae: 0.1080 - val_loss: 0.0171 - val_mae: 0.1050
Epoch 5/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 105s 66ms/step - loss: 0.0195 - mae: 0.1077 - val_loss: 0.0180 - val_mae: 0.1099
Epoch 6/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 105s 66ms/step - loss: 0.0194 - mae: 0.1076 - val_loss: 0.0171 - val_mae: 0.1026
Epoch 7/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 105s 66ms/step - loss: 0.0194 - mae: 0.1075 - val_loss: 0.0177 - val_mae: 0.1084
Epoch 8/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 109s 69ms/step - loss: 0.0194 - mae: 0.1075 - val_loss: 0.0170 - val_mae: 0.1043
Epoch 9/20
1586/

In [71]:
# 1. Predict on scaled test set
y_pred_bilstm_scaled = model_bilstm.predict(X_test)

# 2. Inverse-transform predictions and test actuals back to real Megawatts (MW)
y_pred_bilstm_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))


mae_bilstm = mean_absolute_error(y_test_mw, y_pred_bilstm_mw)
rmse_bilstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_bilstm_mw))
mape_bilstm = np.mean(np.abs((y_test_mw - y_pred_bilstm_mw) / y_test_mw)) * 100
r2_bilstm = r2_score(y_test_mw, y_pred_bilstm_mw)

print("--- Bidirectional LSTM Test Results ---")
print(f"MAE:  {mae_bilstm:.2f} MW")
print(f"RMSE: {rmse_bilstm:.2f} MW")
print(f"MAPE: {mape_bilstm:.2f}%")
print(f"R2:   {r2_bilstm:.4f}")

676/676 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step
--- Bidirectional LSTM Test Results ---
MAE:  5469.22 MW
RMSE: 6697.01 MW
MAPE: 18.79%
R2:   -0.0797
